# GestureX feature engineering

This notebook inspects the two feature representations used by GestureX: the 63 raw MediaPipe coordinates and the 8 geometric features. It uses a real collected sample and the same `src.features` functions as the training and deployment pipeline. It contains no saved data or fabricated experiment output.

## Why compare two representations?

Raw coordinates retain all `21 × 3 = 63` normalized MediaPipe values in landmark order. They can be informative, but framing and apparent hand size may shift between recording sessions. The geometric vector makes a different trade-off: it represents five normalized distances and three joint angles. The experiment evaluates both rather than assuming one is superior.

Before collection, run `python src/collect_data.py`. This notebook stops with a clear error if a valid local CSV is unavailable.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'config.py').is_file():
            return candidate
    raise RuntimeError('Could not find the GestureX repository root.')


ROOT = find_repository_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import INVARIANT_FEATURE_COLUMNS, RAW_DATA_DIR, RAW_FEATURE_COLUMNS
from src.features import extract_invariant_features
from src.landmarks import HAND_CONNECTIONS, unflatten_landmarks


## 1. Load a real sample and display its raw coordinates

Rows are ordered `landmark_0_x`, `landmark_0_y`, `landmark_0_z`, then landmark 1, and so on through landmark 20. Landmark 0 is the wrist.

In [ ]:
csv_paths = sorted(RAW_DATA_DIR.glob('*.csv'))
if not csv_paths:
    raise FileNotFoundError(
        f'No collected CSV found in {RAW_DATA_DIR}. Run python src/collect_data.py first.'
    )

data = pd.concat([pd.read_csv(path) for path in csv_paths], ignore_index=True)
missing = [column for column in RAW_FEATURE_COLUMNS if column not in data.columns]
if missing:
    raise ValueError('CSV is missing required landmark columns: ' + ', '.join(missing))
if data.empty:
    raise ValueError('The collected CSV has no valid samples.')

sample = data.iloc[0]
points = unflatten_landmarks(sample.loc[list(RAW_FEATURE_COLUMNS)].to_numpy(dtype=float))
raw_coordinate_table = pd.DataFrame(points, columns=['x', 'y', 'z'])
raw_coordinate_table.index.name = 'landmark_index'
display(raw_coordinate_table)


## 2. Translation normalization

Let `p_i` be landmark `i` and let `p_0` be the wrist. GestureX first uses the wrist as a reference frame:

```text
p'_i = p_i − p_0
```

A camera translation changes every original coordinate by the same vector, which cancels in `p'_i`. Pairwise distances are already translation-invariant, but explicitly centering all landmarks makes the reference convention clear and keeps future features consistent.

In [ ]:
centered = points - points[0]  # wrist becomes [0, 0, 0]

def draw_hand(ax, coordinates, title):
    for start, end in HAND_CONNECTIONS:
        ax.plot(coordinates[[start, end], 0], coordinates[[start, end], 1], color='tab:blue')
    ax.scatter(coordinates[:, 0], coordinates[:, 1], c='tab:orange', s=35, zorder=2)
    ax.scatter(coordinates[0, 0], coordinates[0, 1], c='crimson', s=60, label='Wrist (0)', zorder=3)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(title)


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
draw_hand(axes[0], points, 'Raw normalized MediaPipe coordinates')
draw_hand(axes[1], centered, 'Wrist-centered coordinates')
axes[1].legend(loc='best')
fig.suptitle(f"Sample gesture: {sample['gesture']}")
plt.tight_layout()
plt.show()


## 3. Scale normalization

A hand closer to the camera often occupies a larger coordinate extent. GestureX uses wrist-to-middle-MCP distance as a per-sample scale:

```text
s = ||p'_middle_mcp||
d_normalized(a, b) = ||p'_a − p'_b|| / s
```

The implementation rejects `s ≤ ε` instead of dividing by a near-zero quantity. Three joint angles are measured in degrees; their cosine calculation is also guarded against zero-length segments.

In [ ]:
MIDDLE_MCP = 9
scale = float(np.linalg.norm(centered[MIDDLE_MCP]))
if scale <= 1e-6:
    raise ValueError('This sample has a near-zero wrist-to-middle-MCP scale and cannot be normalized.')

scale_normalized = centered / scale
print(f'Wrist-to-middle-MCP scale for this actual sample: {scale:.6f}')

fig, ax = plt.subplots(figsize=(5, 5))
draw_hand(ax, scale_normalized, 'Wrist-centered and scale-normalized coordinates')
ax.legend(loc='best')
plt.show()


## 4. Calculate and inspect the eight invariant features

The shared library function returns the required five normalized distances followed by thumb, index, and middle joint angles. The named table below makes the feature order explicit.

In [ ]:
invariant_vector = extract_invariant_features(points)
invariant_table = pd.DataFrame(
    {'feature': INVARIANT_FEATURE_COLUMNS, 'value': invariant_vector}
)
invariant_table


## 5. Deterministic invariance sanity check

This is a mathematical check on the selected real sample, not an accuracy result. Translating all points, or uniformly scaling them about the wrist, should preserve this implementation’s distance ratios and joint angles up to floating-point tolerance.

In [ ]:
translated = points + np.array([0.12, -0.08, 0.03])
scaled_about_wrist = points[0] + 1.7 * (points - points[0])

np.testing.assert_allclose(
    extract_invariant_features(points),
    extract_invariant_features(translated),
    rtol=1e-10,
    atol=1e-10,
)
np.testing.assert_allclose(
    extract_invariant_features(points),
    extract_invariant_features(scaled_about_wrist),
    rtol=1e-10,
    atol=1e-10,
)
print('Translation and uniform-scale invariance checks passed for this sample.')


## Interpretation

Invariant geometry deliberately discards some raw positional detail. Whether that trade-off improves cross-session generalization is an empirical question answered only by the leakage-safe evaluation workflow in `03_model_evaluation.ipynb`. Do not infer accuracy from the plots in this notebook.